# Extracting GHS codes from the EU CLP Regulation
This notebook demonstrates how to extract GHS statement codes from Annex VI of the EU Classification, Labelling and Packaging (CLP) Regulation. The resulting dataset can be used for hazard-based screening or as a component of a global inventory.

**Prerequisites**
- Export the harmonised classifications from the [ECHA CHEM website](https://chem.echa.europa.eu/obligation-lists/clhList) to an Excel file.

## Step 1: Patch the GHS code parser
Each source cell contains a GHS statement code together with descriptive text. The GIC recognises only the GHS code itself, so the descriptive text needs to be stripped before validation.

In [1]:
from gic.fields import ghs_code


def patch_ghs_code(value: str) -> str:
    """Return the GHS code without the accompanying hazard text."""
    try:
        return value.split(", ", 1)[1]
    except IndexError:
        # Leave unexpected values unchanged so they can be logged as errors.
        return value


ghs_code.GHSCodeField.patch = staticmethod(patch_ghs_code)

## Step 2: Import and configure the extractor
Set the input and output paths and map the relevant spreadsheet columns to GIC fields.

In [2]:
from gic import GenericXlsxExtractor

# Initialise the extractor.
extractor = GenericXlsxExtractor()

# Input and output files.
extractor.input_file = "input/Harmonised_List_2026-03-18 11_23_37.xlsx"
extractor.output_file = "output/eu_clp_ghs.jsonl"
extractor.errors_file = "output/eu_clp_ghs_errors.tsv"

# Column-to-field mapping.
extractor.columns = {
    1: "chemical_name",
    2: "ec_number",
    5: "ghs_code",
}

# Add the CLP list tag to each extracted row for potential inventory incorporation.
extractor.constants = {"list_tag": ["EU-CLP"]}

# Sheet layout settings.
extractor.start_row = 2
extractor.subdelimiter = "\n"

## Step 3: Run the extraction and inspect the output
Execute the extractor, write the results to JSONL, and inspect an example record containing multiple GHS codes.

In [3]:
df = extractor.extract()

# Inspect an example record containing multiple GHS codes.
print(df[9].as_dict())

{'chemical_name': ['beryllium oxide'], 'ec_number': ['215-133-1'], 'ghs_code': ['H350i', 'H330', 'H301', 'H335', 'H372', 'H315', 'H319', 'H317'], 'list_tag': ['EU-CLP']}
